In [2]:
!pip install -U ultralytics pyyaml


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.1 MB/s eta 0:00:00


In [28]:
import copy


In [3]:
import torch
from ultralytics import YOLO
import os

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Torch version: 2.8.0+cu126
CUDA available: True
GPU: Tesla P100-PCIE-16GB


In [4]:
# Check YOLOv11 YAML
print("YOLOv11 files:")
!ls /kaggle/input/yolov11

# Check dataset
print("\nPCB dataset files:")
!ls /kaggle/input/pcb-defect-dataset



YOLOv11 files:
yolo11.yaml

PCB dataset files:
pcb-defect-dataset


In [10]:
# List everything Kaggle mounted
!ls /kaggle/input


pcb-defect-dataset  yolov11


In [11]:
!cp /kaggle/input/yolov11/yolo11.yaml /kaggle/working/yolo11.yaml


In [12]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/yolo11.yaml")
model.info()


WARNING ⚠️ no model scale passed. Assuming scale='n'.
YOLO11 summary: 182 layers, 2,624,080 parameters, 2,624,064 gradients, 6.6 GFLOPs


(182, 2624080, 2624064, 6.614336)

In [13]:
# Verify reduced model YAML
!ls /kaggle/working

# Verify dataset
!ls /kaggle/input/pcb-defect-dataset


yolo11.yaml
pcb-defect-dataset


In [14]:
import yaml

with open("/kaggle/working/yolo11.yaml", "r") as f:
    model_cfg = yaml.safe_load(f)

model_cfg.keys()


dict_keys(['nc', 'scales', 'backbone', 'head'])

In [15]:
# Reduce width & depth further (lighter than YOLOv11-n)
model_cfg["scales"]["n"] = [0.33, 0.20, 768]  # depth, width, max_channels


In [16]:
def reduce_blocks(blocks):
    reduced = []
    for layer in blocks:
        layer = layer.copy()
        # layer format: [from, repeats, module, args]
        if isinstance(layer[1], int) and layer[1] > 1:
            layer[1] = max(1, layer[1] // 2)
        reduced.append(layer)
    return reduced

model_cfg["backbone"] = reduce_blocks(model_cfg["backbone"])
model_cfg["head"] = reduce_blocks(model_cfg["head"])


In [17]:
reduced_yaml_path = "/kaggle/working/yolo11_reduced.yaml"

with open(reduced_yaml_path, "w") as f:
    yaml.dump(model_cfg, f)

print("Saved reduced model at:", reduced_yaml_path)


Saved reduced model at: /kaggle/working/yolo11_reduced.yaml


In [18]:
!mv /kaggle/working/yolo11_reduced.yaml /kaggle/working/yolo11n_reduced.yaml


In [19]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/yolo11n_reduced.yaml")
model.info()


YOLO11n_reduced summary: 182 layers, 1,467,900 parameters, 1,467,884 gradients, 4.8 GFLOPs


(182, 1467900, 1467884, 4.8155648)

In [22]:
results = model.train(
    data="/kaggle/input/pcb-defect-dataset/pcb-defect-dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=20,
    cos_lr=True,
    amp=True,
    pretrained=False,
    project="YOLOv11_Reduced",
    name="pcb_yolov11_reduced"
)


Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/input/pcb-defect-dataset/pcb-defect-dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/yolo11n_reduced.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=pcb_yolov11_reduced3, nbs=64, nms=False, opset=None, optimize=

RuntimeError: Dataset '/kaggle/input/pcb-defect-dataset/pcb-defect-dataset/data.yaml' error ❌ Dataset '/kaggle/input/pcb-defect-dataset/pcb-defect-dataset/data.yaml' images not found, missing path '/kaggle/working/pcb-defect-dataset/val'
Note dataset download directory is '/kaggle/working/datasets'. You can update this in '/root/.config/Ultralytics/settings.json'

In [23]:
# Check model YAML in working directory
!ls /kaggle/working

# Check PCB dataset structure
!ls /kaggle/input/pcb-defect-dataset/pcb-defect-dataset


runs  yolo11n_reduced.yaml  yolo11.yaml
data.yaml  test  train	val


In [24]:
with open("/kaggle/working/yolo11.yaml", "r") as f:
    model_cfg = yaml.safe_load(f)

print(model_cfg.keys())


dict_keys(['nc', 'scales', 'backbone', 'head'])


In [26]:
# Original YOLOv11-n ≈ [0.50, 0.25, 1024]
# We reduce further

model_cfg["scales"]["n"] = [0.33, 0.20, 768]


In [29]:
def reduce_blocks(blocks):
    reduced_blocks = []
    for layer in blocks:
        layer = copy.deepcopy(layer)

        # layer format: [from, repeats, module, args]
        if isinstance(layer[1], int) and layer[1] > 1:
            layer[1] = max(1, layer[1] // 2)

        reduced_blocks.append(layer)

    return reduced_blocks


model_cfg["backbone"] = reduce_blocks(model_cfg["backbone"])
model_cfg["head"] = reduce_blocks(model_cfg["head"])


In [30]:
reduced_yaml_path = "/kaggle/working/yolo11n_reduced.yaml"

with open(reduced_yaml_path, "w") as f:
    yaml.dump(model_cfg, f)

print("Saved reduced YOLOv11 model at:", reduced_yaml_path)


Saved reduced YOLOv11 model at: /kaggle/working/yolo11n_reduced.yaml


In [31]:
model = YOLO("/kaggle/working/yolo11n_reduced.yaml")
model.info()


YOLO11n_reduced summary: 182 layers, 1,467,900 parameters, 1,467,884 gradients, 4.8 GFLOPs


(182, 1467900, 1467884, 4.8155648)

In [33]:
!sed -n '1,200p' /kaggle/input/pcb-defect-dataset/pcb-defect-dataset/data.yaml


path: ../pcb-defect-dataset
train: train
val: val
test: test

names:
  0: mouse_bite
  1: spur
  2: missing_hole
  3: short
  4: open_circuit
  5: spurious_copper


In [34]:
import yaml

# Load original data.yaml
with open("/kaggle/input/pcb-defect-dataset/pcb-defect-dataset/data.yaml", "r") as f:
    data_cfg = yaml.safe_load(f)

# Fix paths (ABSOLUTE paths for Kaggle)
data_cfg["train"] = "/kaggle/input/pcb-defect-dataset/pcb-defect-dataset/train"
data_cfg["val"]   = "/kaggle/input/pcb-defect-dataset/pcb-defect-dataset/val"
data_cfg["test"]  = "/kaggle/input/pcb-defect-dataset/pcb-defect-dataset/test"

# Save fixed yaml to working
fixed_data_yaml = "/kaggle/working/pcb_data.yaml"
with open(fixed_data_yaml, "w") as f:
    yaml.dump(data_cfg, f)

print("Saved fixed dataset yaml at:", fixed_data_yaml)


Saved fixed dataset yaml at: /kaggle/working/pcb_data.yaml


In [36]:
!ls /kaggle/input/pcb-defect-dataset/pcb-defect-dataset/train/images | head
!ls /kaggle/input/pcb-defect-dataset/pcb-defect-dataset/val/images | head


light_01_missing_hole_01_1_600.jpg
light_01_missing_hole_01_2_600.jpg
light_01_missing_hole_02_1_600.jpg
light_01_missing_hole_03_1_600.jpg
light_01_missing_hole_04_1_600.jpg
light_01_missing_hole_04_2_600.jpg
light_01_missing_hole_06_1_600.jpg
light_01_missing_hole_06_2_600.jpg
light_01_missing_hole_08_1_600.jpg
light_01_missing_hole_08_3_600.jpg
ls: write error: Broken pipe
light_01_missing_hole_02_2_600.jpg
light_01_missing_hole_03_2_600.jpg
light_01_missing_hole_05_1_600.jpg
light_01_missing_hole_07_2_600.jpg
light_01_missing_hole_09_1_600.jpg
light_01_missing_hole_10_3_600.jpg
light_01_missing_hole_18_3_600.jpg
light_01_missing_hole_19_3_600.jpg
light_01_mouse_bite_02_2_600.jpg
light_01_mouse_bite_09_1_600.jpg
ls: write error: Broken pipe


In [38]:
results = model.train(
    data="/kaggle/working/pcb_data.yaml",   # 🔑 FIXED YAML
    epochs=25,
    imgsz=640,
    batch=16,
    device=0,
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=20,
    cos_lr=True,
    amp=True,
    pretrained=False,
    project="YOLOv11_Reduced",
    name="pcb_yolov11n_reduced"
)


Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/pcb_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/yolo11n_reduced.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=pcb_yolov11n_reduced4, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_

In [46]:
import torch

# High-level summary (already correct)
model.info(verbose=True)

print("\n🔹 Layer-wise parameter details:\n")

total_params = 0

for i, layer in enumerate(model.model.model):  # <-- FIXED
    layer_params = sum(p.numel() for p in layer.parameters())
    total_params += layer_params

    print(
        f"Layer {i:03d} | "
        f"{layer.__class__.__name__:25s} | "
        f"Params: {layer_params}"
    )

print("\n✅ Total parameters (manual count):", total_params)


YOLO11n_reduced summary (fused): 101 layers, 1,425,356 parameters, 0 gradients, 4.5 GFLOPs

🔹 Layer-wise parameter details:

Layer 000 | Conv                      | Params: 448
Layer 001 | Conv                      | Params: 4640
Layer 002 | C3k2                      | Params: 5117
Layer 003 | Conv                      | Params: 28280
Layer 004 | C3k2                      | Params: 17303
Layer 005 | Conv                      | Params: 97448
Layer 006 | C3k2                      | Params: 57200
Layer 007 | Conv                      | Params: 149920
Layer 008 | C3k2                      | Params: 135040
Layer 009 | SPPF                      | Params: 64240
Layer 010 | C2PSA                     | Params: 97600
Layer 011 | Upsample                  | Params: 0
Layer 012 | Concat                    | Params: 0
Layer 013 | C3k2                      | Params: 68302
Layer 014 | Upsample                  | Params: 0
Layer 015 | Concat                    | Params: 0
Layer 016 | C3k2             

In [47]:
import io
import sys

# Capture printed output
buffer = io.StringIO()
sys.stdout = buffer
model.info()
sys.stdout = sys.__stdout__

info_text = buffer.getvalue()
print(info_text)

# Extract GFLOPs
for line in info_text.split("\n"):
    if "GFLOPs" in line:
        print("✅", line.strip())


YOLO11n_reduced summary (fused): 101 layers, 1,425,356 parameters, 0 gradients, 4.5 GFLOPs



In [41]:
metrics = model.val(
    data="/kaggle/working/pcb_data.yaml",
    imgsz=640,
    batch=16,
    device=0,
    plots=False
)

print("\n📊 FINAL METRICS")
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")


Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
YOLO11n_reduced summary (fused): 101 layers, 1,425,356 parameters, 0 gradients, 4.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 86.1±35.3 MB/s, size: 88.8 KB)
val: Scanning /kaggle/input/pcb-defect-dataset/pcb-defect-dataset/val/labels... 802 images, 264 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1066/1066 730.8it/s 1.5s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/pcb-defect-dataset/pcb-defect-dataset/val is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 12.2it/s 5.5s0.1s
                   all       1066       1595      0.949      0.911      0.958      0.502
            mouse_bite        140        280       0.95      0.879      0.958      0.499
                  spur        130        262      0.938      0.844      0.945      0.475
          missing_hole        118        229   

In [42]:
model.val(
    data="/kaggle/working/pcb_data.yaml",
    imgsz=640,
    batch=16,
    device=0,
    plots=True
)


Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 105.5±36.7 MB/s, size: 95.8 KB)
val: Scanning /kaggle/input/pcb-defect-dataset/pcb-defect-dataset/val/labels... 802 images, 264 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1066/1066 909.5it/s 1.2s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/pcb-defect-dataset/pcb-defect-dataset/val is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 9.1it/s 7.4s<0.1s
                   all       1066       1595      0.949      0.911      0.958      0.502
            mouse_bite        140        280       0.95      0.879      0.958      0.499
                  spur        130        262      0.938      0.844      0.945      0.475
          missing_hole        118        229      0.996      0.993      0.995       0.57
                 short        158        327    

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e644f344f20>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [43]:
model.val(
    data="/kaggle/working/pcb_data.yaml",
    imgsz=640,
    batch=16,
    device=0,
    plots=True
)


Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 106.5±23.1 MB/s, size: 110.6 KB)
val: Scanning /kaggle/input/pcb-defect-dataset/pcb-defect-dataset/val/labels... 802 images, 264 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1066/1066 787.3it/s 1.4s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/pcb-defect-dataset/pcb-defect-dataset/val is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 9.0it/s 7.4s<0.2s
                   all       1066       1595      0.949      0.911      0.958      0.502
            mouse_bite        140        280       0.95      0.879      0.958      0.499
                  spur        130        262      0.938      0.844      0.945      0.475
          missing_hole        118        229      0.996      0.993      0.995       0.57
                 short        158        327   

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e644f2d8e30>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [44]:
!ls -R /kaggle/working/runs/detect/YOLOv11_Reduced/pcb_yolov11n_reduced


/kaggle/working/runs/detect/YOLOv11_Reduced/pcb_yolov11n_reduced:
args.yaml  weights

/kaggle/working/runs/detect/YOLOv11_Reduced/pcb_yolov11n_reduced/weights:


In [45]:
import shutil
import os

run_dir = "/kaggle/working/runs/detect/YOLOv11_Reduced/pcb_yolov11n_reduced"
zip_path = "/kaggle/working/YOLOv11_Reduced_PCB_All_Results"

# Create zip
shutil.make_archive(zip_path, 'zip', run_dir)

print("✅ ZIP saved at:", zip_path + ".zip")


✅ ZIP saved at: /kaggle/working/YOLOv11_Reduced_PCB_All_Results.zip
